# ResNet50 Final Accuracy Summary (JSON)

Build a table of final test accuracy (mean +/- stdev across runs) from `resnet50.json`.


In [ ]:
import json
from pathlib import Path
import statistics

import pandas as pd

json_path = Path('resnet50.json')
with json_path.open('r', encoding='utf-8') as f:
    data = json.load(f)


In [ ]:
results = data['results']
rows = []
for dataset_name, heads in results.items():
    for head_name, runs in heads.items():
        final_test = [r['final_test_acc'] for r in runs]
        mean_test = sum(final_test) / len(final_test)
        std_test = statistics.stdev(final_test) if len(final_test) > 1 else 0.0
        rows.append({
            'dataset': dataset_name,
            'head': head_name,
            'mean_test': mean_test,
            'std_test': std_test,
        })

df = pd.DataFrame(rows)
dataset_order = ['FashionMNIST', 'CIFAR10', 'CIFAR100']
head_order = sorted({head for heads in results.values() for head in heads})
df['dataset'] = pd.Categorical(df['dataset'], categories=dataset_order, ordered=True)
df['head'] = pd.Categorical(df['head'], categories=head_order, ordered=True)
df['summary'] = df.apply(
    lambda r: f"{r['mean_test']*100:.2f} ± {r['std_test']*100:.2f}",
    axis=1,
)
table = df.pivot(index='head', columns='dataset', values='summary')
table = table.reindex(index=head_order, columns=dataset_order)
table
